# End-to-End Speech-to-Speech Pipeline

This notebook demonstrates a complete speech-to-speech pipeline by combining:

1. **Whisper (ASR)** - Automatic Speech Recognition to convert speech to text
2. **Text Processing** - Transform or process the transcribed text (could be translation, LLM response, etc.)
3. **Parler-TTS** - Text-to-Speech synthesis to convert processed text back to speech

This creates a full loop where we can take audio input, understand what was said, process or transform the content, and generate new speech output.

In [ ]:
# Install required packages
!pip install transformers parler-tts torch datasets soundfile IPython

## Step 1: Speech-to-Text with Whisper

We'll use OpenAI's Whisper model to transcribe speech into text. Whisper is a powerful ASR model that works well across multiple languages and accents.

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import numpy as np

# Load Whisper ASR pipeline
print("Loading Whisper model...")
device_id = 0 if __import__('torch').cuda.is_available() else -1
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-base",
    device=device_id
)

# Load a sample audio from datasets
print("Loading sample audio...")
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample_audio = ds[0]["audio"]
sample_audio_array = np.array(sample_audio["array"])
sample_audio_sr = sample_audio["sampling_rate"]

# Transcribe the audio
print("Transcribing audio...")
transcription = whisper_pipe(sample_audio_array, generate_kwargs={"language": "english"})
transcribed_text = transcription["text"]

print(f"\nTranscribed text: {transcribed_text}")

## Step 2: Text Processing

This is where you can insert any text transformation logic:
- Translation to another language
- LLM-based response generation
- Text cleanup or formatting
- Sentiment analysis and modification

For this demo, we'll implement a simple text transformation as a placeholder.

In [ ]:
def process_text(text):
    """
    Process the transcribed text.
    This is a placeholder - you can replace this with:
    - LLM API call (OpenAI, Anthropic, etc.)
    - Translation model
    - Custom text processing logic
    """
    # Simple example: add a response prefix
    processed = f"You said: {text}. That's interesting!"
    
    # Alternative examples (commented out):
    # processed = text.upper()  # Convert to uppercase
    # processed = text.replace("hello", "greetings")  # Replace words
    # processed = translate_text(text, target_language="es")  # Translation
    
    return processed

# Process the transcribed text
processed_text = process_text(transcribed_text)
print(f"Processed text: {processed_text}")

## Step 3: Text-to-Speech with Parler-TTS

Now we'll convert the processed text back to speech using Parler-TTS, which allows us to control the voice characteristics through natural language descriptions.

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import torch

# Load Parler-TTS model and tokenizer
print("Loading Parler-TTS model...")
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = ParlerTTSForConditionalGeneration.from_pretrained(
    "parler-tts/parler-tts-mini-v1"
).to(device)
tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-mini-v1")

# Define voice description
description = "A friendly female voice with a clear and moderate pace."

# Generate speech from processed text
print("Generating speech...")
input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(processed_text, return_tensors="pt").input_ids.to(device)

generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_output = generation.cpu().numpy().squeeze()

print(f"Generated audio shape: {audio_output.shape}")
print(f"Sample rate: {model.config.sampling_rate} Hz")

## Full Pipeline

Let's wrap everything into a single function that chains all the steps together for easy reuse.

In [ ]:
def speech_to_speech(audio_array, voice_description="A friendly female voice with a clear and moderate pace."):
    """
    End-to-end speech-to-speech pipeline.
    
    Args:
        audio_array: Numpy audio array
        voice_description: Natural language description of desired output voice
    
    Returns:
        tuple: (transcribed_text, processed_text, audio_output, sample_rate)
    """
    # Step 1: Speech to Text
    print("Step 1: Transcribing speech...")
    transcription = whisper_pipe(audio_array)
    transcribed_text = transcription["text"]
    print(f"  Transcribed: {transcribed_text}")
    
    # Step 2: Process Text
    print("Step 2: Processing text...")
    processed_text = process_text(transcribed_text)
    print(f"  Processed: {processed_text}")
    
    # Step 3: Text to Speech
    print("Step 3: Generating speech...")
    input_ids = tokenizer(voice_description, return_tensors="pt").input_ids.to(device)
    prompt_input_ids = tokenizer(processed_text, return_tensors="pt").input_ids.to(device)
    
    generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
    audio_output = generation.cpu().numpy().squeeze()
    
    print("Pipeline complete!")
    return transcribed_text, processed_text, audio_output, model.config.sampling_rate

# Run the full pipeline
print("Running end-to-end speech-to-speech pipeline...\n")
transcribed, processed, output_audio, sr = speech_to_speech(sample_audio_array)

print(f"\n=== Results ===")
print(f"Input transcription: {transcribed}")
print(f"Processed text: {processed}")
print(f"Output audio shape: {output_audio.shape}")
print(f"Sample rate: {sr} Hz")

## Play Results

Listen to both the input and output audio to verify the pipeline works correctly.

In [ ]:
from IPython.display import Audio, display, HTML

# Display input audio
print("Input Audio:")
display(Audio(sample_audio_array, rate=sample_audio_sr))

print("\nOutput Audio:")
display(Audio(output_audio, rate=sr))

# Optional: Save the output audio to file
output_path = "speech_to_speech_output.wav"
sf.write(output_path, output_audio, sr)
print(f"\nOutput audio saved to: {output_path}")

## Next Steps

This pipeline provides a foundation for building more sophisticated speech-to-speech applications. Here are some ideas for enhancement:

### 1. Add an LLM in the Middle
- Replace the simple text processing with a call to an LLM (GPT-4, Claude, Llama, etc.)
- Create a conversational agent that can respond intelligently to spoken questions
- Example: Voice assistant, language tutor, or interview bot

### 2. Streaming Support
- Implement streaming ASR for real-time transcription
- Use streaming TTS to reduce latency
- Enable more natural conversational interactions

### 3. Multi-Language Support
- Add language detection
- Implement translation between languages
- Use language-specific voice characteristics

### 4. Voice Cloning
- Integrate voice cloning capabilities to match input speaker
- Preserve speaker identity across the pipeline

### 5. Audio Quality Improvements
- Add noise reduction preprocessing
- Implement voice activity detection (VAD)
- Post-process output audio for better quality

### 6. Context and Memory
- Add conversation history tracking
- Implement contextual responses based on previous exchanges
- Build a stateful dialogue system

### 7. Custom Voice Personas
- Create multiple voice profiles with different characteristics
- Allow users to select or customize voice attributes
- Match voice to content (formal vs. casual, etc.)

### 8. Performance Optimization
- Use quantized models for faster inference
- Implement model caching and batching
- Deploy on GPU for real-time performance